## Example 1: Basic Threading Instrumentation Setup and Thread Tracing

In [ ]:
from opentelemetry.instrumentation.threading import ThreadingInstrumentor
from opentelemetry.instrumentation.threading.version import __version__
from opentelemetry.sdk.trace import TracerProvider, Resource
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, BatchSpanProcessor
from opentelemetry import trace
import threading
import time

try:
    # Example 1: Basic threading instrumentation setup and thread tracing
    print("=== OpenTelemetry Threading Instrumentation - Basic Setup ===")
    
    # Configure OpenTelemetry tracing
    resource = Resource.create({"service.name": "threading-demo-app"})
    tracer_provider = TracerProvider(resource=resource)
    
    # Add console exporter to see traces
    console_exporter = ConsoleSpanExporter()
    span_processor = BatchSpanProcessor(console_exporter)
    tracer_provider.add_span_processor(span_processor)
    trace.set_tracer_provider(tracer_provider)
    
    # Get tracer for this example
    tracer = trace.get_tracer("threading-instrumentation-demo", __version__)
    
    # Initialize threading instrumentor
    threading_instrumentor = ThreadingInstrumentor()
    
    print(f"Threading instrumentation version: {__version__}")
    print(f"Instrumentor class: {threading_instrumentor.__class__.__name__}")
    print(f"Dependencies: {list(threading_instrumentor.instrumentation_dependencies())}")
    
    # Check initial instrumentation status
    is_instrumented_before = threading_instrumentor.is_instrumented_by_opentelemetry
    print(f"Instrumented before: {is_instrumented_before}")
    
    # Apply threading instrumentation
    threading_instrumentor.instrument()
    
    # Check status after instrumentation
    is_instrumented_after = threading_instrumentor.is_instrumented_by_opentelemetry
    print(f"Instrumented after: {is_instrumented_after}")
    
    # Define a worker function that will be traced across threads
    def worker_task(task_id, duration):
        """Worker task that runs in a separate thread"""
        with tracer.start_as_current_span(f"worker_task_{task_id}") as span:
            span.set_attribute("task.id", task_id)
            span.set_attribute("task.duration", duration)
            span.set_attribute("thread.name", threading.current_thread().name)
            span.set_attribute("thread.id", threading.get_ident())
            
            print(f"🧵 Task {task_id} starting in thread {threading.current_thread().name}")
            time.sleep(duration)  # Simulate work
            print(f"✅ Task {task_id} completed in {duration}s")
            return f"Task {task_id} result"
    
    # Create and start multiple threads with instrumentation
    print(f"\n🚀 Creating instrumented threads...")
    threads = []
    
    with tracer.start_as_current_span("multi_threading_operation") as parent_span:
        parent_span.set_attribute("operation.type", "parallel_processing")
        parent_span.set_attribute("thread_count", 3)
        
        for i in range(3):
            thread = threading.Thread(
                target=worker_task,
                args=(i + 1, 0.5),  # task_id, duration
                name=f"Worker-{i+1}"
            )
            threads.append(thread)
            thread.start()
            print(f"📤 Started thread: {thread.name}")
        
        # Wait for all threads to complete
        for thread in threads:
            thread.join()
            print(f"📥 Joined thread: {thread.name}")
    
    print(f"✅ Threading instrumentation demonstration completed")
    print(f"All {len(threads)} threads finished with tracing enabled")
    
    # Cleanup
    threading_instrumentor.uninstrument()
    
except Exception as e:
    print(f"✗ Threading instrumentation example FAILED: {type(e).__name__}: {e}")
    raise AssertionError(
        f"opentelemetry_instrumentation_threading example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: Thread Pool and Executor Instrumentation

In [ ]:
import concurrent.futures
from queue import Queue
import random

try:
    # Example 2: Thread pool and executor instrumentation with concurrent futures
    print("=== OpenTelemetry Threading - Thread Pool & Executor Instrumentation ===")
    
    # Re-initialize threading instrumentor for this example
    thread_instrumentor = ThreadingInstrumentor()
    if not thread_instrumentor.is_instrumented_by_opentelemetry:
        thread_instrumentor.instrument()
    
    # Get tracer for thread pool operations
    tracer = trace.get_tracer("thread-pool-demo", __version__)
    
    # Define CPU-intensive task for thread pool
    def cpu_intensive_task(task_id, iterations):
        """Simulate CPU-intensive work in thread pool"""
        with tracer.start_as_current_span(f"cpu_task_{task_id}") as span:
            span.set_attribute("task.id", task_id)
            span.set_attribute("task.iterations", iterations)
            span.set_attribute("executor.type", "ThreadPoolExecutor")
            
            # Simulate CPU work
            result = 0
            for i in range(iterations):
                result += i * random.random()
                if i % 1000 == 0:  # Checkpoint every 1000 iterations
                    span.add_event(f"Progress checkpoint: {i}/{iterations}")
            
            span.set_attribute("task.result", result)
            print(f"🔄 CPU Task {task_id} completed with result: {result:.2f}")
            return result
    
    # Define I/O-bound task simulation
    def io_bound_task(task_id, delay):
        """Simulate I/O-bound work in thread pool"""
        with tracer.start_as_current_span(f"io_task_{task_id}") as span:
            span.set_attribute("task.id", task_id)
            span.set_attribute("task.delay", delay)
            span.set_attribute("executor.type", "ThreadPoolExecutor")
            
            print(f"💾 I/O Task {task_id} starting (delay: {delay}s)")
            time.sleep(delay)  # Simulate I/O wait
            
            span.add_event("I/O operation completed")
            result = f"I/O result for task {task_id}"
            print(f"✅ I/O Task {task_id} completed")
            return result
    
    # Demonstrate ThreadPoolExecutor with instrumentation
    with tracer.start_as_current_span("thread_pool_operations") as pool_span:
        pool_span.set_attribute("pool.max_workers", 4)
        pool_span.set_attribute("operation.type", "mixed_workload")
        
        # Create ThreadPoolExecutor
        with concurrent.futures.ThreadPoolExecutor(max_workers=4, thread_name_prefix="Pool") as executor:
            print(f"🏊 Created thread pool with 4 workers")
            
            # Submit CPU-intensive tasks
            cpu_futures = []
            for i in range(3):
                future = executor.submit(cpu_intensive_task, f"CPU-{i+1}", 5000)
                cpu_futures.append(future)
            
            # Submit I/O-bound tasks
            io_futures = []
            for i in range(3):
                delay = random.uniform(0.2, 0.8)
                future = executor.submit(io_bound_task, f"IO-{i+1}", delay)
                io_futures.append(future)
            
            pool_span.set_attribute("tasks.cpu_count", len(cpu_futures))
            pool_span.set_attribute("tasks.io_count", len(io_futures))
            
            # Wait for CPU tasks to complete
            cpu_results = []
            for future in concurrent.futures.as_completed(cpu_futures, timeout=10):
                try:
                    result = future.result()
                    cpu_results.append(result)
                except Exception as e:
                    print(f"⚠️ CPU task failed: {e}")
                    raise
            
            # Wait for I/O tasks to complete
            io_results = []
            for future in concurrent.futures.as_completed(io_futures, timeout=10):
                try:
                    result = future.result()
                    io_results.append(result)
                except Exception as e:
                    print(f"⚠️ I/O task failed: {e}")
                    raise
            
            pool_span.set_attribute("results.cpu_tasks", len(cpu_results))
            pool_span.set_attribute("results.io_tasks", len(io_results))
            pool_span.add_event("All thread pool tasks completed")
    
    print(f"✅ Thread pool instrumentation completed")
    print(f"Processed {len(cpu_results)} CPU tasks and {len(io_results)} I/O tasks")
    print(f"Average CPU result: {sum(cpu_results) / len(cpu_results) if cpu_results else 0:.2f}")
    
    # Demonstrate producer-consumer pattern with threading
    print(f"\n🏭 Producer-Consumer Pattern with Threading Instrumentation:")
    
    # Create shared queue
    work_queue = Queue(maxsize=10)
    results_queue = Queue()
    
    def producer(item_count):
        """Producer function that adds work items"""
        with tracer.start_as_current_span("producer_thread") as span:
            span.set_attribute("producer.item_count", item_count)
            
            for i in range(item_count):
                work_item = f"WorkItem-{i+1}"
                work_queue.put(work_item)
                span.add_event(f"Produced: {work_item}")
                time.sleep(0.1)  # Simulate production time
            
            # Signal completion
            work_queue.put(None)
            print(f"📤 Producer completed: {item_count} items")
    
    def consumer(consumer_id):
        """Consumer function that processes work items"""
        with tracer.start_as_current_span(f"consumer_thread_{consumer_id}") as span:
            processed_count = 0
            span.set_attribute("consumer.id", consumer_id)
            
            while True:
                try:
                    item = work_queue.get(timeout=2)
                    if item is None:  # Poison pill
                        work_queue.put(None)  # Pass poison pill to other consumers
                        break
                    
                    # Process item
                    processed_result = f"Processed {item} by Consumer-{consumer_id}"
                    results_queue.put(processed_result)
                    processed_count += 1
                    span.add_event(f"Processed: {item}")
                    time.sleep(0.05)  # Simulate processing time
                    
                except:
                    break
            
            span.set_attribute("consumer.processed_count", processed_count)
            print(f"📥 Consumer {consumer_id} completed: {processed_count} items")
    
    # Run producer-consumer with instrumentation
    with tracer.start_as_current_span("producer_consumer_operation"):
        # Start producer thread
        producer_thread = threading.Thread(target=producer, args=(5,), name="Producer")
        producer_thread.start()
        
        # Start consumer threads
        consumer_threads = []
        for i in range(2):
            consumer_thread = threading.Thread(target=consumer, args=(i+1,), name=f"Consumer-{i+1}")
            consumer_threads.append(consumer_thread)
            consumer_thread.start()
        
        # Wait for completion
        producer_thread.join()
        for thread in consumer_threads:
            thread.join()
        
        # Collect results
        results = []
        while not results_queue.empty():
            results.append(results_queue.get())
        
        print(f"✅ Producer-consumer completed: {len(results)} items processed")
    
except Exception as e:
    print(f"✗ Thread pool example FAILED: {type(e).__name__}: {e}")
    raise AssertionError(
        f"opentelemetry_instrumentation_threading example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: Thread Synchronization and Context Propagation

In [ ]:
from opentelemetry import context, baggage
from opentelemetry.trace.propagation.tracecontext import TraceContextTextMapPropagator
import threading
from threading import Lock, Condition, Semaphore

try:
    # Example 3: Thread synchronization and context propagation patterns
    print("=== OpenTelemetry Threading - Synchronization & Context Propagation ===")
    
    # Ensure threading instrumentation is active
    if not ThreadingInstrumentor().is_instrumented_by_opentelemetry:
        ThreadingInstrumentor().instrument()
    
    tracer = trace.get_tracer("sync-context-demo", __version__)
    
    # Shared resources for synchronization demos
    shared_counter = 0
    counter_lock = Lock()
    condition = Condition()
    semaphore = Semaphore(2)  # Allow max 2 threads
    shared_data = []
    
    # Context propagation demo - parent sets context, children inherit
    def synchronized_worker(worker_id, increment_count):
        """Worker that uses synchronization primitives with context propagation"""
        global shared_counter, shared_data
        
        # Create child span that inherits parent context
        with tracer.start_as_current_span(f"sync_worker_{worker_id}") as span:
            span.set_attribute("worker.id", worker_id)
            span.set_attribute("worker.increment_count", increment_count)
            
            # Get baggage from context (propagated from parent)
            request_id = baggage.get_baggage("request.id")
            user_id = baggage.get_baggage("user.id")
            span.set_attribute("context.request_id", request_id or "unknown")
            span.set_attribute("context.user_id", user_id or "unknown")
            
            # Demonstrate semaphore usage
            print(f"🔒 Worker {worker_id} waiting for semaphore...")
            with semaphore:
                span.add_event("Acquired semaphore")
                print(f"🔓 Worker {worker_id} acquired semaphore")
                
                # Critical section with lock
                for i in range(increment_count):
                    with counter_lock:
                        old_value = shared_counter
                        shared_counter += 1
                        shared_data.append(f"Worker-{worker_id}-{i+1}")
                        
                        span.add_event(f"Incremented counter: {old_value} -> {shared_counter}")
                        time.sleep(0.01)  # Simulate work
                
                print(f"✅ Worker {worker_id} completed {increment_count} increments")
                span.add_event("Released semaphore")
    
    # Condition variable demo with context
    def condition_waiter(waiter_id):
        """Thread that waits on condition variable"""
        with tracer.start_as_current_span(f"condition_waiter_{waiter_id}") as span:
            span.set_attribute("waiter.id", waiter_id)
            
            with condition:
                print(f"⏳ Waiter {waiter_id} waiting for condition...")
                span.add_event("Waiting on condition variable")
                condition.wait(timeout=5)  # Wait up to 5 seconds
                
                print(f"🔔 Waiter {waiter_id} woke up!")
                span.add_event("Condition signaled")
    
    def condition_notifier():
        """Thread that signals condition variable"""
        with tracer.start_as_current_span("condition_notifier") as span:
            time.sleep(1)  # Let waiters start waiting
            
            with condition:
                print(f"📢 Notifier signaling condition...")
                span.add_event("Notifying all waiters")
                condition.notify_all()
    
    # Main context propagation demo
    with tracer.start_as_current_span("thread_sync_operation") as parent_span:
        # Set baggage for context propagation
        ctx = baggage.set_baggage("request.id", "req-12345")
        ctx = baggage.set_baggage("user.id", "user-67890", context=ctx)
        
        parent_span.set_attribute("operation.type", "synchronized_threading")
        parent_span.set_attribute("thread_count", 4)
        
        print(f"🌍 Parent context set with request.id and user.id")
        
        # Use context token to propagate context to child threads
        token = context.attach(ctx)
        
        try:
            # Start synchronized workers
            worker_threads = []
            for i in range(4):
                thread = threading.Thread(
                    target=synchronized_worker,
                    args=(i + 1, 3),  # worker_id, increment_count
                    name=f"SyncWorker-{i+1}"
                )
                worker_threads.append(thread)
                thread.start()
            
            # Start condition variable demo threads
            condition_threads = []
            for i in range(2):
                waiter_thread = threading.Thread(
                    target=condition_waiter,
                    args=(i + 1,),
                    name=f"CondWaiter-{i+1}"
                )
                condition_threads.append(waiter_thread)
                waiter_thread.start()
            
            # Start notifier thread
            notifier_thread = threading.Thread(target=condition_notifier, name="CondNotifier")
            notifier_thread.start()
            
            # Wait for all threads
            for thread in worker_threads:
                thread.join()
            for thread in condition_threads:
                thread.join()
            notifier_thread.join()
            
            parent_span.set_attribute("final_counter_value", shared_counter)
            parent_span.set_attribute("data_items_count", len(shared_data))
            parent_span.add_event("All synchronized threads completed")
            
        finally:
            context.detach(token)
    
    print(f"✅ Thread synchronization completed")
    print(f"Final counter value: {shared_counter}")
    print(f"Shared data items: {len(shared_data)}")
    print(f"Sample data: {shared_data[:5] if shared_data else []}")
    
    # Demonstrate deadlock prevention patterns
    print(f"\n🔐 Deadlock Prevention Patterns:")
    
    # Two locks for deadlock demo
    lock_a = Lock()
    lock_b = Lock()
    
    def ordered_locking_task(task_id, reverse_order=False):
        """Task that acquires locks in consistent order to prevent deadlock"""
        with tracer.start_as_current_span(f"ordered_locking_{task_id}") as span:
            span.set_attribute("task.id", task_id)
            span.set_attribute("task.reverse_order", reverse_order)
            
            if reverse_order:
                # This could cause deadlock, but we'll use timeout
                try:
                    if lock_b.acquire(timeout=2):
                        span.add_event("Acquired lock_b")
                        time.sleep(0.1)
                        if lock_a.acquire(timeout=2):
                            span.add_event("Acquired lock_a")
                            print(f"🔄 Task {task_id} got both locks (reverse order)")
                            time.sleep(0.1)
                            lock_a.release()
                            span.add_event("Released lock_a")
                        lock_b.release()
                        span.add_event("Released lock_b")
                except:
                    span.add_event("Timeout acquiring locks - deadlock prevented")
                    print(f"⚠️ Task {task_id} timed out - deadlock prevented")
            else:
                # Consistent ordering prevents deadlock
                with lock_a:
                    span.add_event("Acquired lock_a")
                    time.sleep(0.1)
                    with lock_b:
                        span.add_event("Acquired lock_b")
                        print(f"✅ Task {task_id} got both locks (proper order)")
                        time.sleep(0.1)
                        span.add_event("Released lock_b")
                    span.add_event("Released lock_a")
    
    # Test deadlock prevention
    deadlock_threads = []
    for i in range(2):
        thread = threading.Thread(
            target=ordered_locking_task,
            args=(i + 1, i == 1),  # Second thread uses reverse order
            name=f"DeadlockTest-{i+1}"
        )
        deadlock_threads.append(thread)
        thread.start()
    
    for thread in deadlock_threads:
        thread.join()
    
    print(f"✅ Deadlock prevention demonstration completed")
    
except Exception as e:
    print(f"✗ Thread synchronization example FAILED: {type(e).__name__}: {e}")
    raise AssertionError(
        f"opentelemetry_instrumentation_threading example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

## Example 4: Async/Threading Integration and Performance Monitoring

In [ ]:
import asyncio
from threading import Thread, Event
import queue
import psutil
import os

try:
    # Example 4: Async/threading integration and performance monitoring
    print("=== OpenTelemetry Threading - Async Integration & Performance ===")
    
    # Ensure instrumentation is active
    if not ThreadingInstrumentor().is_instrumented_by_opentelemetry:
        ThreadingInstrumentor().instrument()
    
    tracer = trace.get_tracer("async-threading-perf", __version__)
    
    # Performance monitoring data
    thread_metrics = {
        "threads_created": 0,
        "threads_completed": 0,
        "total_execution_time": 0,
        "memory_usage_mb": 0
    }
    
    # Async-to-sync bridge pattern
    class AsyncBridge:
        """Bridge between async and sync code using threading"""
        
        def __init__(self):
            self.loop = None
            self.thread = None
            self.ready = Event()
            
        def start_async_loop(self):
            """Start async event loop in separate thread"""
            with tracer.start_as_current_span("async_loop_thread") as span:
                span.set_attribute("bridge.type", "async_loop")
                
                self.loop = asyncio.new_event_loop()
                asyncio.set_event_loop(self.loop)
                self.ready.set()
                
                print("🔄 Async event loop started in thread")
                span.add_event("Event loop initialized")
                
                try:
                    self.loop.run_forever()
                except:
                    pass
                finally:
                    self.loop.close()
                    span.add_event("Event loop closed")
        
        def run_async_in_thread(self, coro):
            """Run async coroutine in the thread's event loop"""
            if not self.thread or not self.thread.is_alive():
                self.thread = Thread(target=self.start_async_loop, name="AsyncLoop")
                self.thread.start()
                self.ready.wait()  # Wait for loop to be ready
            
            future = asyncio.run_coroutine_threadsafe(coro, self.loop)
            return future.result(timeout=10)
        
        def shutdown(self):
            """Shutdown the async loop"""
            if self.loop:
                self.loop.call_soon_threadsafe(self.loop.stop)
                self.thread.join()
    
    # Async task that will run in separate thread
    async def async_worker_task(task_id, delay):
        """Async task that simulates I/O work"""
        # Note: This creates span in the async thread
        with tracer.start_as_current_span(f"async_task_{task_id}") as span:
            span.set_attribute("task.id", task_id)
            span.set_attribute("task.delay", delay)
            span.set_attribute("execution.type", "async_in_thread")
            
            print(f"🚀 Async task {task_id} starting (delay: {delay}s)")
            await asyncio.sleep(delay)
            
            result = f"Async result {task_id}"
            span.set_attribute("task.result", result)
            print(f"✅ Async task {task_id} completed")
            return result
    
    # Performance monitoring worker
    def performance_monitor(duration, metrics_dict):
        """Monitor thread and system performance"""
        with tracer.start_as_current_span("performance_monitor") as span:
            span.set_attribute("monitor.duration", duration)
            
            start_time = time.time()
            process = psutil.Process(os.getpid())
            
            while time.time() - start_time < duration:
                # Get current metrics
                memory_mb = process.memory_info().rss / 1024 / 1024
                thread_count = threading.active_count()
                
                metrics_dict["memory_usage_mb"] = memory_mb
                metrics_dict["active_threads"] = thread_count
                
                span.add_event(f"Memory: {memory_mb:.1f}MB, Threads: {thread_count}")
                time.sleep(0.5)
            
            span.set_attribute("final.memory_mb", metrics_dict["memory_usage_mb"])
            span.set_attribute("final.thread_count", thread_count)
    
    # Mixed workload: sync threads + async tasks
    with tracer.start_as_current_span("mixed_workload_operation") as main_span:
        main_span.set_attribute("workload.type", "mixed_sync_async")
        
        # Start performance monitoring
        perf_thread = Thread(
            target=performance_monitor,
            args=(8, thread_metrics),
            name="PerfMonitor"
        )
        perf_thread.start()
        thread_metrics["threads_created"] += 1
        
        # Create async bridge
        bridge = AsyncBridge()
        
        # Define CPU-bound sync task
        def cpu_bound_task(task_id, iterations):
            """CPU-bound task in regular thread"""
            with tracer.start_as_current_span(f"cpu_bound_{task_id}") as span:
                span.set_attribute("task.id", task_id)
                span.set_attribute("task.iterations", iterations)
                
                start_time = time.time()
                result = sum(i * i for i in range(iterations))
                duration = time.time() - start_time
                
                thread_metrics["total_execution_time"] += duration
                span.set_attribute("task.duration", duration)
                span.set_attribute("task.result", result)
                
                print(f"⚡ CPU task {task_id} completed in {duration:.3f}s")
                return result
        
        # Start CPU-bound threads
        cpu_threads = []
        for i in range(3):
            thread = Thread(
                target=cpu_bound_task,
                args=(i + 1, 50000),
                name=f"CPUWorker-{i+1}"
            )
            cpu_threads.append(thread)
            thread.start()
            thread_metrics["threads_created"] += 1
        
        # Start async tasks via bridge
        async_results = []
        for i in range(3):
            try:
                result = bridge.run_async_in_thread(
                    async_worker_task(i + 1, 1.0)
                )
                async_results.append(result)
            except Exception as e:
                print(f"⚠️ Async task {i+1} failed: {e}")
                raise
        
        # Wait for CPU threads
        for thread in cpu_threads:
            thread.join()
            thread_metrics["threads_completed"] += 1
        
        # Wait for performance monitor
        perf_thread.join()
        thread_metrics["threads_completed"] += 1
        
        # Shutdown async bridge
        bridge.shutdown()
        
        main_span.set_attribute("results.async_tasks", len(async_results))
        main_span.set_attribute("results.cpu_tasks", len(cpu_threads))
        main_span.set_attribute("performance.peak_memory_mb", thread_metrics["memory_usage_mb"])
    
    # Performance summary
    print(f"\n📊 Performance Summary:")
    print(f"Threads created: {thread_metrics['threads_created']}")
    print(f"Threads completed: {thread_metrics['threads_completed']}")
    print(f"Total CPU execution time: {thread_metrics['total_execution_time']:.2f}s")
    print(f"Peak memory usage: {thread_metrics['memory_usage_mb']:.1f}MB")
    print(f"Async results: {len(async_results)}")
    
    # Thread lifecycle monitoring
    print(f"\n🔍 Thread Lifecycle Monitoring:")
    active_threads = threading.enumerate()
    for thread in active_threads:
        print(f"  🧵 {thread.name}: {'alive' if thread.is_alive() else 'finished'}")
    
    threading_instrumentor.uninstrument()
    print(f"✅ Async/threading integration and performance monitoring completed")
    
except Exception as e:
    print(f"✗ Async/threading integration example FAILED: {type(e).__name__}: {e}")
    raise AssertionError(
        f"opentelemetry_instrumentation_threading example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: Production Threading Patterns and Best Practices

In [ ]:
import signal
from collections import deque
import json

try:
    # Example 5: Production threading patterns and best practices
    print("=== OpenTelemetry Threading - Production Patterns & Best Practices ===")
    
    # Ensure instrumentation is active for production patterns
    threading_instrumentor = ThreadingInstrumentor()
    if not threading_instrumentor.is_instrumented_by_opentelemetry:
        threading_instrumentor.instrument()
    
    tracer = trace.get_tracer("production-threading", __version__)
    
    # Production thread pool manager
    class ProductionThreadManager:
        """Production-grade thread pool manager with observability"""
        
        def __init__(self, max_workers=4, queue_size=100):
            self.max_workers = max_workers
            self.queue_size = queue_size
            self.work_queue = queue.Queue(maxsize=queue_size)
            self.result_queue = queue.Queue()
            self.workers = []
            self.shutdown_event = Event()
            self.stats = {
                "tasks_submitted": 0,
                "tasks_completed": 0,
                "tasks_failed": 0,
                "workers_started": 0,
                "workers_stopped": 0
            }
            
        def start(self):
            """Start worker threads"""
            with tracer.start_as_current_span("thread_manager_start") as span:
                span.set_attribute("manager.max_workers", self.max_workers)
                span.set_attribute("manager.queue_size", self.queue_size)
                
                for i in range(self.max_workers):
                    worker = Thread(
                        target=self._worker_loop,
                        args=(i + 1,),
                        name=f"ProdWorker-{i+1}"
                    )
                    worker.daemon = True  # Allow clean shutdown
                    worker.start()
                    self.workers.append(worker)
                    self.stats["workers_started"] += 1
                
                print(f"🏭 Started {len(self.workers)} production workers")
                span.add_event(f"Started {len(self.workers)} workers")
        
        def _worker_loop(self, worker_id):
            """Main worker loop with error handling"""
            with tracer.start_as_current_span(f"worker_loop_{worker_id}") as span:
                span.set_attribute("worker.id", worker_id)
                tasks_processed = 0
                
                while not self.shutdown_event.is_set():
                    try:
                        # Get task with timeout
                        task_func, args, kwargs = self.work_queue.get(timeout=1)
                        
                        if task_func is None:  # Shutdown signal
                            break
                        
                        # Execute task with span
                        with tracer.start_as_current_span(f"task_execution_{worker_id}") as task_span:
                            task_span.set_attribute("worker.id", worker_id)
                            task_span.set_attribute("task.type", task_func.__name__)
                            
                            try:
                                result = task_func(*args, **kwargs)
                                self.result_queue.put(("success", result))
                                self.stats["tasks_completed"] += 1
                                tasks_processed += 1
                                task_span.set_status(trace.Status(trace.StatusCode.OK))
                                
                            except Exception as e:
                                self.result_queue.put(("error", str(e)))
                                self.stats["tasks_failed"] += 1
                                task_span.set_status(trace.Status(
                                    trace.StatusCode.ERROR, str(e)
                                ))
                                print(f"⚠️ Worker {worker_id} task failed: {e}")
                        
                        self.work_queue.task_done()
                        
                    except queue.Empty:
                        continue  # Check shutdown event
                    except Exception as e:
                        print(f"⚠️ Worker {worker_id} error: {e}")
                
                span.set_attribute("worker.tasks_processed", tasks_processed)
                self.stats["workers_stopped"] += 1
                print(f"🔚 Worker {worker_id} stopped after {tasks_processed} tasks")
        
        def submit(self, task_func, *args, **kwargs):
            """Submit task to worker pool"""
            try:
                self.work_queue.put((task_func, args, kwargs), timeout=1)
                self.stats["tasks_submitted"] += 1
                return True
            except queue.Full:
                print("⚠️ Task queue is full")
                return False
        
        def shutdown(self, timeout=5):
            """Graceful shutdown of thread pool"""
            with tracer.start_as_current_span("thread_manager_shutdown") as span:
                span.set_attribute("shutdown.timeout", timeout)
                
                # Signal shutdown
                self.shutdown_event.set()
                
                # Send shutdown signals to workers
                for _ in self.workers:
                    try:
                        self.work_queue.put((None, (), {}), timeout=0.1)
                    except queue.Full:
                        pass
                
                # Wait for workers to finish
                for worker in self.workers:
                    worker.join(timeout=timeout / len(self.workers))
                    
                span.add_event("All workers shutdown")
                print(f"🛑 Thread pool shutdown completed")
        
        def get_stats(self):
            """Get production statistics"""
            return self.stats.copy()
    
    # Production task definitions
    def cpu_production_task(task_id, complexity):
        """Production CPU task with error handling"""
        if complexity <= 0:
            raise ValueError("Invalid complexity parameter")
            
        result = sum(i ** 2 for i in range(complexity))
        time.sleep(0.1)  # Simulate processing time
        return f"CPU-{task_id}: {result}"
    
    def io_production_task(task_id, duration):
        """Production I/O task with timeout handling"""
        if duration > 5:
            raise ValueError("Duration too long for production")
            
        time.sleep(duration)
        return f"I/O-{task_id}: completed in {duration}s"
    
    # Production deployment simulation
    with tracer.start_as_current_span("production_deployment") as deploy_span:
        deploy_span.set_attribute("deployment.environment", "production")
        deploy_span.set_attribute("deployment.version", "1.0.0")
        
        # Create and start production thread manager
        manager = ProductionThreadManager(max_workers=3, queue_size=20)
        manager.start()
        
        # Submit production workload
        print(f"📋 Submitting production workload...")
        
        # Submit CPU tasks with valid parameters
        for i in range(5):
            success = manager.submit(cpu_production_task, i + 1, 1000)
            assert success, f"CPU task {i+1} was not accepted by the work queue"
        
        # Submit I/O tasks
        for i in range(5):
            duration = 0.2 + (i * 0.1)
            success = manager.submit(io_production_task, i + 1, duration)
            assert success, f"I/O task {i+1} was not accepted by the work queue"
        
        # Monitor progress
        time.sleep(3)  # Let tasks complete
        
        # Collect results
        results = []
        errors = []
        while not manager.result_queue.empty():
            try:
                status, result = manager.result_queue.get_nowait()
                if status == "success":
                    results.append(result)
                else:
                    errors.append(result)
            except queue.Empty:
                break
        
        # Get final statistics
        stats = manager.get_stats()
        
        # Graceful shutdown
        manager.shutdown()
        
        deploy_span.set_attribute("results.successful_tasks", len(results))
        deploy_span.set_attribute("results.failed_tasks", len(errors))
        deploy_span.set_attribute("stats.tasks_submitted", stats["tasks_submitted"])
        deploy_span.set_attribute("stats.workers_started", stats["workers_started"])
    
    # Production statistics and monitoring
    print(f"\n📊 Production Statistics:")
    print(json.dumps(stats, indent=2))
    
    print(f"\n✅ Results: {len(results)} successful tasks")
    # Every submitted task is valid (complexity > 0, duration <= 5 s), so none may fail
    assert not errors, f"{len(errors)} production tasks failed: {errors}"
    if results:
        print(f"Sample results: {results[:3]}")
    
    # Best practices summary
    best_practices = [
        "✅ Use daemon threads for background workers",
        "✅ Implement graceful shutdown with timeouts",
        "✅ Add comprehensive error handling and logging",
        "✅ Monitor thread pool statistics and performance",
        "✅ Use bounded queues to prevent memory issues",
        "✅ Implement proper context propagation across threads"
    ]
    
    print(f"\n📋 Production Threading Best Practices:")
    for practice in best_practices:
        print(f"  {practice}")
    
    # Cleanup instrumentation
    threading_instrumentor.uninstrument()
    
    print(f"\n✅ Production threading patterns and best practices demonstrated")
    
except Exception as e:
    print(f"✗ Production patterns example FAILED: {type(e).__name__}: {e}")
    raise AssertionError(
        f"opentelemetry_instrumentation_threading example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e